# Final Model 1: Feature Engineering 강화 버전

이 노트북은 `submission_holiday_dinner_stronger_annotated.ipynb`를 기반으로 **피처 엔지니어링(Feature Engineering)**을 추가한 버전입니다.

**추가된 내용:**
1. **메뉴 선호도 점수 (Menu Score):** 과거 데이터를 바탕으로 메뉴의 인기도를 수치화하여 추가.
2. **요일별 최근 추세 (Lag Feature):** 최근 4주간 해당 요일의 평균 참여율을 반영.
3. **계절성 (Season):** 월 정보를 바탕으로 계절 변수 추가.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from xgboost import XGBRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색
train_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\train_median.csv",
    r"./data/train_median.csv",
    r"../data/train_median.csv",
]
test_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\test.csv",
    r"./data/test.csv",
    r"../data/test.csv",
]
sample_sub_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv",
    r"./data/sample_submission.csv",
    r"../data/sample_submission.csv",
]
weather_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\weather.csv",
    r"./data/weather.csv",
    r"../data/weather.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
test_path = find_existing_path(test_candidates)
sample_sub_path = find_existing_path(sample_sub_candidates)
weather_path = find_existing_path(weather_candidates)

if train_path is None or test_path is None or sample_sub_path is None or weather_path is None:
    # Fallback to local if paths not found (for portability)
    train = pd.read_csv("train_median.csv", encoding="utf-8-sig")
    test = pd.read_csv("test.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv("sample_submission.csv", encoding="utf-8-sig")
    weather = pd.read_csv("weather.csv", encoding="utf-8-sig")
else:
    train = pd.read_csv(train_path, encoding="utf-8-sig")
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    sample_submission = pd.read_csv(sample_sub_path, encoding="utf-8-sig")
    weather = pd.read_csv(weather_path, encoding="utf-8-sig")

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 날씨 전처리
def find_col(cols, candidates):
    for cand in candidates:
        for c in cols:
            if cand.lower() == c.lower():
                return c
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

date_col = find_col(weather.columns, ["일시", "date", "날짜"])
temp_col = find_col(weather.columns, ["기온", "평균기온", "temperature", "avg_temp"])
rain_col = find_col(weather.columns, ["강수량", "rainfall", "precipitation", "rain"])

weather = weather[[date_col, temp_col, rain_col]].copy()
weather = weather.rename(columns={date_col:"일자", temp_col:"기온", rain_col:"강수량"})
weather["일자"] = pd.to_datetime(weather["일자"])
weather["기온"] = pd.to_numeric(weather["기온"], errors="coerce").interpolate().bfill().ffill()
weather["강수량"] = pd.to_numeric(weather["강수량"], errors="coerce").fillna(0)
weather["is_rain"] = (weather["강수량"] > 0).astype(int)
weather["is_hot"] = (weather["기온"] >= 28).astype(int)
weather["is_cold"] = (weather["기온"] <= 5).astype(int)

train = train.merge(weather, on="일자", how="left")
test = test.merge(weather, on="일자", how="left")

for df_ in [train, test]:
    for c in ["기온","강수량","is_rain","is_hot","is_cold"]:
        if c.startswith("is_"):
            df_[c] = df_[c].fillna(0)
        else:
            df_[c] = df_[c].interpolate().bfill().ffill()

In [ ]:
# 기본 피처 생성 + [추가] 계절/Lag 피처
def add_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)

    # [추가] 계절 변수
    df["season"] = df["월"].map({3:0, 4:0, 5:0, 6:1, 7:1, 8:1, 9:2, 10:2, 11:2, 12:3, 1:3, 2:3})

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])

    prev_gap = df["일자"].diff().dt.days.fillna(1)
    next_gap = df["일자"].shift(-1).sub(df["일자"]).dt.days.fillna(1)
    df["holiday_after"] = (prev_gap >= 2).astype(int)
    df["holiday_before"] = (next_gap >= 2).astype(int)
    
    # [추가] 샌드위치 데이? (앞뒤로 연휴 기운이 있을 때)
    df["is_sandwich"] = (df["holiday_before"] & df["holiday_after"]).astype(int)

    return df

train = add_features(train)
test = add_features(test)

train["중식참여율"] = (train["중식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)
train["석식참여율"] = (train["석식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)

# [추가] Lag Feature (최근 4주 요일별 평균)
# Train의 마지막 4주 데이터를 기준으로 요일별 평균을 계산해 Test에 매핑
last_month_start = train["일자"].max() - pd.Timedelta(weeks=4)
recent_train = train[train["일자"] > last_month_start]

lunch_weekday_rank = recent_train.groupby("요일")["중식참여율"].mean()
dinner_weekday_rank = recent_train.groupby("요일")["석식참여율"].mean()

train["lunch_weekday_avg"] = train["요일"].map(lunch_weekday_rank)
test["lunch_weekday_avg"] = test["요일"].map(lunch_weekday_rank)
train["dinner_weekday_avg"] = train["요일"].map(dinner_weekday_rank)
test["dinner_weekday_avg"] = test["요일"].map(dinner_weekday_rank)

# [추가] 메뉴 점수화 (Target Encoding)
def get_menu_score(texts, targets):
    word_score = {}
    word_count = {}
    
    # 학습
    for text, target in zip(texts, targets):
        if pd.isna(text): continue
        text = re.sub(r"[^가-힣a-zA-Z]", " ", str(text))
        words = text.split()
        for w in words:
            if len(w) < 2: continue # 한 글자 제외
            word_score[w] = word_score.get(w, 0) + target
            word_count[w] = word_count.get(w, 0) + 1
            
    final_score = {k: v / word_count[k] for k, v in word_score.items() if word_count[k] >= 5}
    mean_val = np.mean(list(final_score.values()))
    
    return final_score, mean_val

lunch_word_map, lunch_base = get_menu_score(train["중식메뉴"], train["중식참여율"])
dinner_word_map, dinner_base = get_menu_score(train["석식메뉴"], train["석식참여율"])

def apply_menu_score(text, word_map, base_val):
    if pd.isna(text): return base_val
    text = re.sub(r"[^가-힣a-zA-Z]", " ", str(text))
    words = text.split()
    scores = [word_map.get(w, base_val) for w in words if len(w) >= 2]
    return np.mean(scores) if scores else base_val

train["lunch_menu_score"] = train["중식메뉴"].apply(lambda x: apply_menu_score(x, lunch_word_map, lunch_base))
test["lunch_menu_score"] = test["중식메뉴"].apply(lambda x: apply_menu_score(x, lunch_word_map, lunch_base))
train["dinner_menu_score"] = train["석식메뉴"].apply(lambda x: apply_menu_score(x, dinner_word_map, dinner_base))
test["dinner_menu_score"] = test["석식메뉴"].apply(lambda x: apply_menu_score(x, dinner_word_map, dinner_base))

In [ ]:
# 피처 목록 업데이트 (새로 만든 피처 포함)
lunch_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_fri","days_to_month_end","is_month_end_part",
    "season", "is_sandwich", "lunch_weekday_avg", "lunch_menu_score"
]
dinner_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_wed","has_overtime","log_overtime",
    "season", "is_sandwich", "dinner_weekday_avg", "dinner_menu_score"
]

In [ ]:
# 모델 학습 (XGBoost)
params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "gamma": 0.0,
}

X_train_lunch = train[lunch_features]
X_test_lunch = test[lunch_features]
y_lunch = train["중식참여율"]
X_train_dinner = train[dinner_features]
X_test_dinner = test[dinner_features]
y_dinner = train["석식참여율"]

xgb_lunch = XGBRegressor(objective="reg:squarederror", random_state=42, **params)
xgb_dinner = XGBRegressor(objective="reg:squarederror", random_state=42, **params)

xgb_lunch.fit(X_train_lunch, y_lunch)
xgb_dinner.fit(X_train_dinner, y_dinner)

base_pred_lunch_ratio = np.clip(xgb_lunch.predict(X_test_lunch), 0, 1.5)
base_pred_dinner_ratio = np.clip(xgb_dinner.predict(X_test_dinner), 0, 1.5)

In [ ]:
# Weak Corrections (기존 로직 유지 - 날씨, 연휴 보정)
# 메뉴 Correction은 'menu_score'가 피처로 들어갔으므로 제외하거나 약하게 유지할 수 있으나,
# 여기서는 피처로 반영했으므로 중복을 피하기 위해 Correction은 제외합니다.

weather_lunch_signal = (
    0.010 * test["is_rain"].values
    - 0.006 * test["is_hot"].values
    + 0.004 * test["is_cold"].values
)
weather_dinner_signal = (
    0.004 * test["is_rain"].values
    + 0.003 * test["is_cold"].values
)

holiday_lunch_signal = (
    -0.004 * test["holiday_before"].values
    + 0.003 * test["holiday_after"].values
)
holiday_dinner_signal = (
    -0.005 * test["holiday_before"].values
    + 0.003 * test["holiday_after"].values
)

final_pred_lunch_ratio = np.clip(base_pred_lunch_ratio + weather_lunch_signal + holiday_lunch_signal, 0, 1.5)
final_pred_dinner_ratio = np.clip(base_pred_dinner_ratio + weather_dinner_signal + holiday_dinner_signal, 0, 1.5)

pred_lunch_count = np.clip(final_pred_lunch_ratio * test["식사가능자수"].values, 0, None)
pred_dinner_count = np.clip(final_pred_dinner_ratio * test["식사가능자수"].values, 0, None)

submission = sample_submission.copy()
if "중식계" in submission.columns:
    submission["중식계"] = pred_lunch_count
    submission["석식계"] = pred_dinner_count
else:
    submission.iloc[:, 1] = pred_lunch_count
    submission.iloc[:, 2] = pred_dinner_count

submission.to_csv("finalmodel1_submission.csv", index=False, encoding="utf-8-sig")
print("저장 완료: finalmodel1_submission.csv")